In [ ]:
"""
avenue11_phase2_sundman_truncation.py
====================================================================
PHASE 2: THE TRUNCATION OF SUNDMAN
Proving that the Infinite Series is Bounded by Modulo-9 Saturation
====================================================================
Key Principles:
1. Sundman's solution requires infinite terms for exact convergence
2. In Finitism, the series truncates when states hit d_max = 9
3. The number of terms is bounded by: N_terms < log_9(state_space)
4. This renders the "infinite series" physically impossible

By Néstor E. Ramos


"""
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("PHASE 2: THE TRUNCATION OF SUNDMAN (RESOLUTION RESTORED)")
print("=" * 80)

# =============================================================================
# 1. PARAMETERS (Corregidos con Espacio de Estados Total de 6 Dimensiones)
# =============================================================================
L = 137         # Spatial limit
T_MAX = 500     # Time steps
G = 2.5         # Gravitational constant
m = [3, 5, 3]   # Masses
N_BODIES = 3
N_e = 15        # Quantum substrate resolution

# Rango dinámico real observado de la velocidad para el cálculo del espacio (V_max ~ 20)
V_MAX = 20

# CORRECCIÓN DIMENSIONAL: Espacio total de fase (3 Posiciones + 3 Velocidades)
STATE_SPACE_SIZE = ((L * N_e) * (V_MAX * N_e)) ** N_BODIES
MAX_INFO_TERMS = np.log(STATE_SPACE_SIZE) / np.log(9)

print(f"Resolucion del Sustrato (N_e) = {N_e}")
print(f"Total State Space Size (6D Phase Space) = {STATE_SPACE_SIZE:.2e}")
print(f"Max Informational Terms (log_9(State Space)) = {MAX_INFO_TERMS:.2f}")


# =============================================================================
# 2. SIMULATION ENGINES
# =============================================================================
def run_continuous(x_init, v_init, T, G, m):
    """Proxy for Infinite Sundman Series."""
    x = np.array(x_init, dtype=float)
    v = np.array(v_init, dtype=float)
    traj_x = np.zeros((T, N_BODIES))
    unique_states_cont = set()

    for t in range(T):
        traj_x[t] = x
        state_hash = tuple(np.round(x, decimals=6))
        unique_states_cont.add(state_hash)

        a = np.zeros(N_BODIES)
        for i in range(N_BODIES):
            for j in range(i+1, N_BODIES):
                dx = x[j] - x[i]
                if abs(dx) < 1: dx = np.sign(dx) * 1.0 if dx != 0 else 1.0
                F = G * m[i] * m[j] / (dx**2)
                dir_ij = np.sign(dx)
                a[i] += dir_ij * F / m[i]
                a[j] -= dir_ij * F / m[j]
        v += a
        x += v
    return traj_x, unique_states_cont

def run_finitist(x_init, v_init, T, L, G, m, N_e):
    """Finitist Quantized Physics with Balanced Rounding."""
    x = np.array(x_init, dtype=float)
    v = np.array(v_init, dtype=float)
    traj_x = np.zeros((T, N_BODIES))

    quantum = 1.0 / N_e
    seen = set()
    cumulative_unique = []
    state_history = []

    for t in range(T):
        # Sincronización al pixel del hardware
        x = np.round(x / quantum) * quantum
        v = np.round(v / quantum) * quantum

        traj_x[t] = x
        state_hash = tuple(np.round(x, 4).tolist() + np.round(v, 4).tolist())
        seen.add(state_hash)
        cumulative_unique.append(len(seen))
        state_history.append(state_hash)

        a = np.zeros(N_BODIES)
        for i in range(N_BODIES):
            for j in range(i+1, N_BODIES):
                dx = (x[j] - x[i] + L/2) % L - L/2
                if abs(dx) < 1.0:
                    dx = np.sign(dx) * 1.0 if dx != 0 else 1.0

                F = G * m[i] * m[j] / (dx**2)
                dir_ij = np.sign(dx)
                a[i] += dir_ij * F / m[i]
                a[j] -= dir_ij * F / m[j]

        v += a
        x = (x + v) % L

    # Calcular periodo real del ciclo límite
    period = T
    for i in range(T-1, 0, -1):
        if state_history[i] == state_history[0]:
            period = i
            break
    return traj_x, cumulative_unique, period

# =============================================================================
# 3. RUN SIMULATIONS
# =============================================================================
# =============================================================================
# 3. RUN SIMULATIONS (Asimetria inyectada para activar al cuerpo verde)
# =============================================================================
# Rompemos la simetria: El cuerpo verde ya no esta en 68, lo ponemos en 66
x0 = [58.0, 66.0, 78.0]
# Le damos una leve velocidad inicial al centro para perturbar el equilibrio
v0 = [0.5, 0.1, -0.5]

x_cont, unique_cont = run_continuous(x0, v0, T_MAX, G, m)
x_fin, cumulative_unique, period = run_finitist(x0, v0, T_MAX, L, G, m, N_e)


# =============================================================================
# 4. VISUALIZATION
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Avenue 11: The Truncation of Sundman via Information Saturation',
             fontsize=14, fontweight='bold', y=0.98)

colors = ['blue', 'green', 'red']

# Panel 1: Continuo
for i in range(N_BODIES):
    axes[0, 0].plot(x_cont[:, i], color=colors[i], linewidth=1.5)
axes[0, 0].set_title('1. Continuous Dynamics (Sundman Proxy)\n(Infinite Precision, Unbounded Space)')
axes[0, 0].grid(True, alpha=0.3)

# Panel 2: Finitista (¡Ahora con órbitas reales vivas!)
for i in range(N_BODIES):
    axes[0, 1].plot(x_fin[:, i], color=colors[i], linewidth=2)
axes[0, 1].set_title('2. Finitist Dynamics\n(Pixelated Space-Time via N_e, Modulo-137 Ring)', color='red')
axes[0, 1].set_ylim(-5, 142)
axes[0, 1].grid(True, alpha=0.3)

# -----------------------------------------------------------------------------
# Panel 3: Saturación Espacial (Límites Forzados para Mostrar la Cota Real)
# -----------------------------------------------------------------------------
ax3 = axes[1, 0] # Aseguramos la indexación correcta de la matriz de subplots
ax3.plot(range(T_MAX), cumulative_unique, 'r-', linewidth=2.5, label='Visited Unique States')

# Dibujar el techo de hardware real (Línea negra discontinua)
ax3.axhline(y=STATE_SPACE_SIZE, color='black', linestyle='--', linewidth=2,
            label=f'Theoretical Max State Space ({STATE_SPACE_SIZE:.1e})')

# Cota secundaria de información
ax3.axhline(y=MAX_INFO_TERMS, color='green', linestyle=':', linewidth=3,
            label=f'Information Bound: log_9(S) = {MAX_INFO_TERMS:.2f}')

ax3.set_title('3. Information Space Saturation\n(The Bound of the Infinite Series)', fontsize=11, fontweight='bold')
ax3.set_xlabel('Time Step')
ax3.set_ylabel('Unique States (Log Scale)')

# --- EL CORRECAMINOS GRÁFICO: FORZAR ESCALA Y LÍMITES ---
ax3.set_yscale('log')

# Forzamos el rango desde 1 (10^0) hasta un paso más arriba de la capacidad total (STATE_SPACE_SIZE * 10)
ax3.set_ylim(1, STATE_SPACE_SIZE * 10)

ax3.legend(loc='lower right', fontsize=9)
ax3.grid(True, alpha=0.3, which="both", ls=":")



# Panel 4: Interpretación Física
axes[1, 1].axis('off')
textstr = (
    "PHYSICAL INTERPRETATION (Computational Finitism):\n\n"
    "1. The Sundman Illusion:\n"
    "   The classical infinite series assumes continuous space\n"
    "   and infinite decimal precision. It is physically uncomputable.\n\n"
    "2. Truncation via Saturation:\n"
    "   By pixelating the sustratum to steps of 1/N_e, the space\n"
    "   of unique states becomes strictly finite.\n\n"
    "3. The Limit Cycle Point:\n"
    "   As seen in Panel 3, the cumulative unique states curve\n"
    "   bends and saturates horizontally. The system enters a\n"
    "   deterministic limit cycle. The infinite series truncates.\n\n"
    "CONCLUSION:\n"
    "The 3-body chaos is bounded by information saturation.\n"
    "The universe limits the terms of the series by hardware design."
)
axes[1, 1].text(0.05, 0.95, textstr, transform=axes[1, 1].transAxes, fontsize=11,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9), family='monospace')

plt.tight_layout()
plt.savefig('avenue11_phase2_sundman_truncation.png', dpi=1200)
print("✓ ¡Gráfico científico corregido y legible exportado con éxito!")
plt.show()


PHASE 2: THE TRUNCATION OF SUNDMAN (RESOLUTION RESTORED)
Resolucion del Sustrato (N_e) = 15
Total State Space Size (6D Phase Space) = 2.34e+17
Max Informational Terms (log_9(State Space)) = 18.20
